[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/ner/wip-bioclinicalbert-crf.ipynb)

> **WIP — INCOMPLETE:** Training fails at model initialization due to a `post_init()` conflict between the custom `BioClinicalBertCRF` layer and HuggingFace's `PreTrainedModel` infrastructure (`'list' object has no attribute 'keys'`). The architecture design is complete; the compatibility issue with `_tied_weights_keys` was not resolved. Left for reference.

# Description

To load this from remote:
```
from transformers import pipeline

# It will download once and cache it for future use
ner_pipe = pipeline("ner", model="alexd063/bio-clinicalbert-finetuned-medmentions")
results = ner_pipe("Patient shows symptoms of acute appendicitis.")
print(results)
```

In [ ]:
%%capture
!pip install transformers datasets evaluate seqeval accelerate torch numpy pytorch-crf
!pip install --upgrade transformers

In [ ]:
import torch
import evaluate
import numpy as np
import torch.nn as nn
import pandas as pd
from torchcrf import CRF

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    BertPreTrainedModel,
    BertModel
)
from huggingface_hub import notebook_login

In [ ]:
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [ ]:
PUSH_TO_HUB = True  # Set to False to skip login and hub push

if PUSH_TO_HUB:
    notebook_login()

In [ ]:
import torch

class Config:
    MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
    DATASET_ID = "Ben10x/MedMentions-MTI881-NER"
    HUB_REPO_ID = "alexd063/bio-clinicalbert-finetuned-medmentions-v2"

    MAX_LENGTH = 512
    BATCH_SIZE = 16
    EPOCHS = 3
    LEARNING_RATE = 6e-5
    WARMUP_RATIO = 0.10
    WEIGHT_DECAY = 0.01

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()

In [ ]:
import time
notebook_start_time = time.time()

In [ ]:
# ==========================================
# 1. Load Dataset & Extract Labels
# ==========================================
# MedMentions-ZS is available directly on the Hub
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")

# The dataset stores tags as strings (e.g., 'B-T062', 'I-T062', 'O').
# We need to extract all unique tags to create our label mappings.
# Note: Adjust the column names 'tokens' and 'ner_tags' if the dataset uses different keys.
unique_tags = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        unique_tags.update(row)

label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# ==========================================
# 2. Tokenization & Label Alignment
# ==========================================
model_id = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_and_align_labels(examples):
    # Tokenize the pre-split words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )

    labels = []
    for i, label_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens (like [CLS] and [SEP]) map to None. We set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_sequence[word_idx]])
            # For subsequent tokens of the same word, also assign -100 (or you can assign the I- tag).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply to train, validation, and test splits
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# ==========================================
# 3. Metrics
# ==========================================

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": np.round(results["overall_precision"], 3),
        "recall": np.round(results["overall_recall"], 3),
        "f1": np.round(results["overall_f1"], 3),
        "accuracy": np.round(results["overall_accuracy"], 3),
    }

In [ ]:
# ==========================================
# 4.0 Preparing the training
# ==========================================

# Calculate Exact Class Weights for Training
all_tags_flat = [tag for sublist in dataset['train']['ner_tags'] for tag in sublist]
tag_counts = pd.Series(all_tags_flat).value_counts()

num_classes = len(label_list)
total_samples = tag_counts.sum()

# Formula: Total Samples / (Number of Classes * Samples per Class)
# ----------------------------------------------------
# V1:
# class_weights_dict = {
#     tag: total_samples / (num_classes * count)
#     for tag, count in tag_counts.items()
# }
# ----------------------------------------------------
# V2:
class_weights_dict = {
    tag: np.round(np.exp(-(num_classes * count)/total_samples) * 100)
    for tag, count in tag_counts.items()
}
class_weights_dict['O'] = np.float64(0.5)
# ----------------------------------------------------

# Map to the label2id order so we can pass it directly to PyTorch
class_weights = [class_weights_dict.get(id2label[i], 1.0) for i in range(num_classes)]

print("Top 5 highest weights (Rarest classes):", sorted(class_weights, reverse=True)[:5])
print("Weight for 'O' class:", class_weights_dict.get('O', 1.0))

In [ ]:
class_weights_dict_v2 = {
    # tag: np.log(total_samples / (num_classes * count))
    tag: np.round(np.exp(-(num_classes * count)/total_samples) * 100)
    for tag, count in tag_counts.items()
}
class_weights_dict_v2['O'] = np.float64(0.5)

In [ ]:
# compile class weights dict with original frequency counts ie tag: (weight, count)
{
    tag: (class_weights_dict_v2.get(tag, 1.0), count)
    for tag, count in tag_counts.items()
}



In [ ]:
# compile class weights dict with original frequency counts ie tag: (weight, count)
{
    tag: (class_weights_dict.get(tag, 1.0), count)
    for tag, count in tag_counts.items()
}



In [ ]:
# ==========================================
# 4.1 Training
# ==========================================


# class BioClinicalBertCRF(BertPreTrainedModel):
#     def __init__(self, config):
#         super().__init__(config)
#         self.num_labels = config.num_labels
#         self.bert = BertModel(config)
#         self.dropout = nn.Dropout(config.hidden_dropout_prob)
#         self.classifier = nn.Linear(config.hidden_size, config.num_labels)
#         self.crf = CRF(config.num_labels, batch_first=True)
#         self.init_weights()

#     def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
#         # 1. Get BERT representations
#         outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
#         sequence_output = outputs[0]
#         sequence_output = self.dropout(sequence_output)

#         # 2. Get emission scores (logits) from the linear layer
#         emissions = self.classifier(sequence_output)

#         # 3. Create a mask for the CRF (CRF expects ByteTensor or BoolTensor)
#         # We must mask out the padding so the CRF ignores it
#         mask = attention_mask.bool()

#         if labels is not None:
#             # During training: calculate negative log-likelihood loss
#             # CRF library expects labels to be (batch, seq_len)
#             log_likelihood = self.crf(emissions, labels, mask=mask, reduction='mean')
#             return (-log_likelihood, emissions) # Loss must be positive for optimization
#         else:
#             # During inference: use Viterbi algorithm to find best path
#             prediction = self.crf.decode(emissions, mask=mask)
#             return prediction

class BioClinicalBertCRF(BertPreTrainedModel):
    # 1. Define the base model prefix so the library knows which
    # attribute holds the BERT weights during loading.
    base_model_prefix = "bert"

    # 2. Define tied weights keys as an empty list to satisfy the internal check
    _tied_weights_keys = []

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels

        # Initialize the backbone
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize CRF
        self.crf = CRF(config.num_labels, batch_first=True)

        # 3. Use post_init() instead of init_weights().
        # This handles weight initialization and internal HF metadata.
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = outputs[0]
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        # CRF requires a mask (1 for real tokens, 0 for padding)
        mask = attention_mask.bool()

        if labels is not None:
            # Training mode: return loss
            log_likelihood = self.crf(logits, labels, mask=mask, reduction='mean')
            return {"loss": -log_likelihood, "logits": logits}
        else:
            # Inference mode: return best tag sequence
            prediction = self.crf.decode(logits, mask=mask)
            return prediction


class CRFTrainer(Trainer):
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        # We override this to handle the list output from CRF.decode()
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            outputs = model(**inputs)
            if isinstance(outputs, tuple):
                loss, logits = outputs
            else:
                loss = None
                logits = outputs # This is the list of sequences from crf.decode()

        if prediction_loss_only:
            return (loss, None, None)

        # Convert list of lists (decoded tags) to a padded tensor for compatibility
        max_len = inputs['input_ids'].shape[1]
        padded_predictions = [seq + [-100] * (max_len - len(seq)) for seq in logits]
        predictions = torch.tensor(padded_predictions)

        return (loss, predictions, inputs['labels'])


training_args = TrainingArguments(
    output_dir="./bio-clinicalbert-medmentions",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,

    # Optimizer & Learning Rate settings
    learning_rate=6e-5,               # Peak learning rate after warmup
    lr_scheduler_type="cosine",       # Activates the Cosine decay
    warmup_ratio=0.10,                # Spends the first 10% of total steps warming up
    weight_decay=0.01,                # Regularization to prevent overfitting

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    push_to_hub=True,
    hub_model_id="alexd063/bio-clinicalbert-finetuned-medmentions-v2",
)

config = AutoConfig.from_pretrained(
    "alexd063/bio-clinicalbert-finetuned-medmentions",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model = BioClinicalBertCRF.from_pretrained(
    "alexd063/bio-clinicalbert-finetuned-medmentions",
    config=config
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = CRFTrainer(
    model=model,
    class_weights=class_weights,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Start Fine-tuning
trainer.train()

# Save the final model
trainer.save_model("./bio-clinicalbert-medmentions-final")

In [ ]:
notebook_end_time = time.time()
elapsed = notebook_end_time - notebook_start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total notebook execution time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")

In [ ]:
tokenized_datasets

In [ ]:
# ==========================================
# 5. Assessment on the Test Set
# ==========================================

# Use the trainer to evaluate the test split
test_results = trainer.evaluate(tokenized_datasets["test"])

# Print the final metrics
print("--- Test Set Evaluation Results ---")
for key, value in test_results.items():
    # Formatting key names for better readability (e.g., eval_f1 -> f1)
    metric_name = key.replace("eval_", "")
    print(f"{metric_name:10}: {value:.4f}")

# Optional: Get detailed predictions if you need to inspect specific examples
# predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
# print("\nDetailed Test Metrics:", metrics)

In [ ]:
import numpy as np
import pandas as pd
from seqeval.metrics import classification_report

# 1. Get predictions from your trainer
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# 2. Remove ignored index (-100) and convert IDs to labels
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

# 3. Generate the per-class report
# output_dict=True allows us to convert it into a DataFrame immediately
report = classification_report(true_labels, true_predictions, output_dict=True)
df_report = pd.DataFrame(report).transpose().reset_index()
df_report.rename(columns={'index': 'Entity_Type'}, inplace=True)

# Filter out the summary rows to see just the individual classes
class_report = df_report[~df_report['Entity_Type'].isin(['micro avg', 'macro avg', 'weighted avg', 'samples avg'])]
print(class_report.sort_values(by='f1-score', ascending=False).head(10))

In [ ]:
class_report.sort_values(by='f1-score', ascending=False)

In [ ]:
tokenized_datasets["validation"]['ner_tags']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot Top 10 vs Bottom 10 entities by F1-score
top_10 = class_report.sort_values(by='f1-score', ascending=False).head(10)
bottom_10 = class_report.sort_values(by='f1-score', ascending=True).head(10)
plot_data = pd.concat([top_10, bottom_10])

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_data, x='f1-score', y='Entity_Type', palette='viridis')
plt.title('Top and Bottom Performing Medical Entities (F1-Score)')
plt.axvline(class_report['f1-score'].mean(), color='red', linestyle='--', label='Mean F1')
plt.legend()
plt.show()

In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub(
        repo_id=cfg.HUB_REPO_ID,
        dataset=cfg.DATASET_ID
    )
    tokenizer.push_to_hub(repo_id=cfg.HUB_REPO_ID)